In [ ]:
from transformers import BertForQuestionAnswering
from transformers import BertTokenizer
import transformers 
import torch
# print versions to make sure you have them installed in your environment
print(transformers.__version__)
print(torch.__version__)
# following prints the environment name. 
import sys
print(sys.executable)

# following code gets the HF_TOKEN saved in .env file via config.py. That helps avoid warnings such as 
# Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
# It also helps avoid throttling issues
import os
from dotenv import load_dotenv
load_dotenv()

import config

if config.HF_TOKEN:
    print("Huggingface TOKEN loaded successfully")
else:
    print("Huggingface TOKEN not found")

my_hf_token=config.HF_TOKEN
os.environ["HF_TOKEN"] = my_hf_token  

## Load model and tokenizer

In [ ]:
# uncased treats upper and lower case
model_name = "bert-large-uncased-whole-word-masking-finetuned-squad"

# suppress warnings
import logging
transformers.logging.set_verbosity_error() # suppresses the warnings

In [ ]:
# set HF_TOKEN=hf_yourTokenHere (get the token first if you don't have it. https://huggingface.co/settings/tokens)
# conda activate llms_course_env_311
# python -c "from transformers import BertForQuestionAnswering; BertForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')"

model = BertForQuestionAnswering.from_pretrained(model_name)

In [ ]:
tokenizer = BertTokenizer.from_pretrained(model_name)

## Embeddings

In [ ]:
# example question and text containing the answer
question = "When was the first dvd released?"
answer_document = "The first DVD (Digital Versatile Disc) was released on March 24, 1997. It was a movie titled 'Twister' and was released in Japan. DVDs quickly gained popularity as a replacement for VHS tapes and became a common format for storing and distributing digital video and data."

In [ ]:
encoding = tokenizer(
    text=question,
    text_pair=answer_document,
    return_tensors="pt" # must add this
)

# # alternate code
# encoding = tokenizer(
#     text=question,
#     text_pair=answer_document,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=512
# )

# # Access the outputs the same way as before
# input_ids = encoding["input_ids"]
# token_type_ids = encoding["token_type_ids"]
# attention_mask = encoding["attention_mask"]

In [ ]:
print(encoding)

In [ ]:
inputs = encoding['input_ids']
sentence_embedding = encoding['token_type_ids']
# inputs is a 2D tensor (shape [1, seq_len]) due to return_tensors="pt", but convert_ids_to_tokens expects a 1D list. 
# You need to squeeze out the batch dimension.
tokens = tokenizer.convert_ids_to_tokens(inputs[0])  # ← add [0]

In [ ]:
tokenizer.decode(101)

In [ ]:
tokenizer.decode(102)

In [ ]:
output = model(input_ids=inputs, token_type_ids=sentence_embedding)

# # alternate code if you want to be explicit
# output = model(
#     input_ids=encoding['input_ids'],
#     token_type_ids=encoding['token_type_ids']
# )
# print(output)

## Model output

In [ ]:
start_index = torch.argmax(output.start_logits)
end_index = torch.argmax(output.end_logits)

print(start_index)
print(end_index)

In [ ]:
answer = ' '.join(tokens[start_index:end_index+1])
print(answer)

In [ ]:
import matplotlib as plt
import seaborn as sns

In [ ]:
s_scores = output.start_logits.detach().numpy().flatten()
e_scores = output.end_logits.detach().numpy().flatten()

In [ ]:
token_labels = []
for (i, token) in enumerate(tokens):
    token_labels.append('{:} - {:>2}'.format(token, i))

In [ ]:
# The two plots visualize different parts of the QA model's output:

# - **`s_scores`** — the model's confidence for each token being the **start** of the answer span
# - **`e_scores`** — the model's confidence for each token being the **end** of the answer span

# The BERT QA model outputs two score distributions across all tokens. The answer is extracted by finding the token with the highest start score and the token with the highest end score, then taking everything in between as the answer.

# So if your question is *"When was the first DVD released?"* and the answer is *"March 24, 1997"*:
# - The **start plot** would show a spike at the token *"March"*
# - The **end plot** would show a spike at the token *"1997"*

# Together they pinpoint the exact answer span in the document.

ax = sns.barplot(x=token_labels, y=s_scores)
ax.tick_params(axis='x', rotation=90)
ax.grid(True)

In [ ]:
ax = sns.barplot(x=token_labels, y=e_scores)
ax.tick_params(axis='x', rotation=90)
ax.grid(True)

## Question Answering

In [ ]:
sunset_motors_context = "Sunset Motors is a renowned automobile dealership that has been a cornerstone of the automotive industry since its establishment in 1978. Located in the picturesque town of Crestwood, nestled in the heart of California's scenic Central Valley, Sunset Motors has built a reputation for excellence, reliability, and customer satisfaction over the past four decades. Founded by visionary entrepreneur Robert Anderson, Sunset Motors began as a humble, family-owned business with a small lot of used cars. However, under Anderson's leadership and commitment to quality, it quickly evolved into a thriving dealership offering a wide range of vehicles from various manufacturers. Today, the dealership spans over 10 acres, showcasing a vast inventory of new and pre-owned cars, trucks, SUVs, and luxury vehicles. One of Sunset Motors' standout features is its dedication to sustainability. In 2010, the dealership made a landmark decision to incorporate environmentally friendly practices, including solar panels to power the facility, energy-efficient lighting, and a comprehensive recycling program. This commitment to eco-consciousness has earned Sunset Motors recognition as an industry leader in sustainable automotive retail. Sunset Motors proudly offers a diverse range of vehicles, including popular brands like Ford, Toyota, Honda, Chevrolet, and BMW, catering to a wide spectrum of tastes and preferences. In addition to its outstanding vehicle selection, Sunset Motors offers flexible financing options, allowing customers to secure affordable loans and leases with competitive interest rates."
print(sunset_motors_context)

In [ ]:
def faq_bot(question):

    context = sunset_motors_context
    input_ids = tokenizer.encode(question, context)
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    sep_idx = input_ids.index(tokenizer.sep_token_id)
    num_seg_a = sep_idx+1
    num_seg_b = len(input_ids) - num_seg_a
    segment_ids = [0]*num_seg_a + [1]*num_seg_b
    output = model(torch.tensor([input_ids]), token_type_ids = torch.tensor([segment_ids]))
    answer_start = torch.argmax(output.start_logits)
    answer_end = torch.argmax(output.end_logits)
    if answer_end >= answer_start:
        answer = ' '.join(tokens[answer_start:answer_end+1])
    else:
        print("I don't know how to answer this question, can you ask another one?")
    corrected_answer = ''
    for word in answer.split():
        if word[0:2] == '##':
            corrected_answer += word[2:]
        else:
            corrected_answer += ' ' + word
    return corrected_answer

In [ ]:
# Following question can't be answered because the model does not know how to handle it. You would need to split your questions.
# faq_bot("What city the dealership is located? Include name of the State as well.")
# it can be easily fixed by returning the message"I don't know how to answer this question, can you ask another one?" instead of just printing it. 
# the code fails afterwards. answer does not have a value. 
faq_bot("Where is the dealershiplocated?") 

In [ ]:
faq_bot("what State the dealership is located?") 

In [ ]:
faq_bot("what make of cars are available?")

In [ ]:
faq_bot("how large is the dealership?") 

## RoBERTa and DistilBERT

In [ ]:
# Before running the following or using this model follow these steps:
# set HF_TOKEN=hf_yourTokenHere (get the token first if you don't have it. https://huggingface.co/settings/tokens)
# conda activate llms_course_env_311
# python -c "from transformers import RobertaTokenizer, RobertaModel; RobertaTokenizer.from_pretrained('roberta-base'); RobertaModel.from_pretrained('roberta-base')"
from transformers import RobertaTokenizer, RobertaModel
model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaModel.from_pretrained(model_name)

In [ ]:
# Before running the following or using this model follow these steps:
# set HF_TOKEN=hf_yourTokenHere (get the token first if you don't have it. https://huggingface.co/settings/tokens)
# conda activate llms_course_env_311
# python -c "from transformers import DistilBertTokenizer, DistilBertModel; DistilBertTokenizer.from_pretrained('distilbert-base-uncased'); DistilBertModel.from_pretrained('distilbert-base-uncased')"
from transformers import  DistilBertTokenizer, DistilBertModel 
model_name = "distilbert-base-uncased"
tokenizer =  DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertModel.from_pretrained(model_name)